# What Drives the Price of a Used Car?
### Practical Application II — CRISP-DM Framework

![CRISP-DM](images/crisp.png)

---

## 1. Business Understanding

### Objective
A used car dealership wants to understand what factors drive the price of a used car. By identifying the key price drivers, the dealership can make better inventory decisions — stocking cars that consumers value and pricing them competitively.

### Key Business Questions
- Which features (year, mileage, condition, type, etc.) most strongly predict used car price?
- Are there specific manufacturers, fuel types, or vehicle types that command a premium?
- What actionable recommendations can we provide to help the dealership optimize its inventory?

### Success Criteria
- Build a regression model that accurately predicts used car prices
- Identify and rank the most important price-driving features
- Provide clear, actionable recommendations for a non-technical audience

## 2. Imports & Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.metrics import mean_squared_error
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer

import warnings
warnings.filterwarnings('ignore')

# Plot styling
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 100
plt.rcParams['font.size'] = 12

## 3. Data Understanding

In [ ]:
# Load the dataset
df = pd.read_csv('data/vehicles.csv')

print(f'Dataset shape: {df.shape}')
print(f'Columns: {df.columns.tolist()}')
df.head()

In [ ]:
# Data types and basic info
df.info()

In [ ]:
# Summary statistics
df.describe()

In [ ]:
# Missing value analysis
missing = df.isnull().sum().sort_values(ascending=False)
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
missing_df = missing_df[missing_df['Missing Count'] > 0]

fig, ax = plt.subplots(figsize=(10, 5))
sns.barplot(x=missing_df['Missing %'], y=missing_df.index, palette='Reds_r', ax=ax)
ax.set_title('Missing Values by Column (%)', fontsize=14, fontweight='bold')
ax.set_xlabel('Missing %')
ax.set_ylabel('')
for i, v in enumerate(missing_df['Missing %']):
    ax.text(v + 0.3, i, f'{v:.1f}%', va='center')
plt.tight_layout()
plt.show()

print(missing_df)

### 3.1 Price Distribution

In [ ]:
# Raw price distribution vs filtered range
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].hist(df['price'], bins=100, color='steelblue', edgecolor='white')
axes[0].set_title('Raw Price Distribution', fontweight='bold')
axes[0].set_xlabel('Price ($)')
axes[0].set_ylabel('Count')

df_viz = df[(df['price'] > 500) & (df['price'] < 100000)]
axes[1].hist(df_viz['price'], bins=100, color='teal', edgecolor='white')
axes[1].set_title('Price Distribution ($500 – $100K)', fontweight='bold')
axes[1].set_xlabel('Price ($)')
axes[1].set_ylabel('Count')

plt.tight_layout()
plt.show()

print('Price percentiles:')
print(df['price'].quantile([0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]))

### 3.2 Categorical Feature Distributions

In [ ]:
cat_cols = ['manufacturer', 'fuel', 'transmission', 'condition', 'type', 'drive']

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for i, col in enumerate(cat_cols):
    top = df[col].value_counts().head(10)
    sns.barplot(x=top.values, y=top.index, ax=axes[i], palette='viridis')
    axes[i].set_title(f'{col.title()} Distribution', fontweight='bold')
    axes[i].set_xlabel('Count')
    axes[i].set_ylabel('')

plt.suptitle('Categorical Feature Distributions', fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

### 3.3 Price by Key Categories

In [ ]:
df_clean_viz = df[(df['price'] > 500) & (df['price'] < 80000)].copy()

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Price by condition
condition_order = ['new', 'like new', 'excellent', 'good', 'fair', 'salvage']
condition_data = df_clean_viz[df_clean_viz['condition'].isin(condition_order)]
sns.boxplot(data=condition_data, x='condition', y='price', order=condition_order,
            palette='RdYlGn_r', ax=axes[0, 0])
axes[0, 0].set_title('Price by Condition', fontweight='bold')
axes[0, 0].set_xlabel('Condition')
axes[0, 0].set_ylabel('Price ($)')
axes[0, 0].tick_params(axis='x', rotation=15)

# Price by fuel type
sns.boxplot(data=df_clean_viz.dropna(subset=['fuel']), x='fuel', y='price',
            palette='Set2', ax=axes[0, 1])
axes[0, 1].set_title('Price by Fuel Type', fontweight='bold')
axes[0, 1].set_xlabel('Fuel Type')
axes[0, 1].set_ylabel('Price ($)')

# Price by drive type
sns.boxplot(data=df_clean_viz.dropna(subset=['drive']), x='drive', y='price',
            palette='Set1', ax=axes[1, 0])
axes[1, 0].set_title('Price by Drive Type', fontweight='bold')
axes[1, 0].set_xlabel('Drive Type')
axes[1, 0].set_ylabel('Price ($)')

# Price by vehicle type
top_types = df_clean_viz['type'].value_counts().head(8).index
type_data = df_clean_viz[df_clean_viz['type'].isin(top_types)]
sns.boxplot(data=type_data, x='type', y='price', order=top_types,
            palette='tab10', ax=axes[1, 1])
axes[1, 1].set_title('Price by Vehicle Type', fontweight='bold')
axes[1, 1].set_xlabel('Vehicle Type')
axes[1, 1].set_ylabel('Price ($)')
axes[1, 1].tick_params(axis='x', rotation=30)

plt.suptitle('Price Distribution by Key Categories', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Median price by top manufacturers
top_manufacturers = df_clean_viz['manufacturer'].value_counts().head(15).index
mfr_price = (df_clean_viz[df_clean_viz['manufacturer'].isin(top_manufacturers)]
             .groupby('manufacturer')['price']
             .median()
             .sort_values(ascending=False))

fig, ax = plt.subplots(figsize=(12, 5))
sns.barplot(x=mfr_price.index, y=mfr_price.values, palette='coolwarm', ax=ax)
ax.set_title('Median Price by Manufacturer (Top 15)', fontweight='bold', fontsize=14)
ax.set_xlabel('Manufacturer')
ax.set_ylabel('Median Price ($)')
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Price vs continuous features
df_scatter = df_clean_viz.dropna(subset=['year', 'odometer']).sample(5000, random_state=42)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(df_scatter['year'], df_scatter['price'], alpha=0.3, color='steelblue', s=10)
axes[0].set_title('Price vs. Year', fontweight='bold')
axes[0].set_xlabel('Year')
axes[0].set_ylabel('Price ($)')

axes[1].scatter(df_scatter['odometer'], df_scatter['price'], alpha=0.3, color='tomato', s=10)
axes[1].set_title('Price vs. Odometer', fontweight='bold')
axes[1].set_xlabel('Odometer (miles)')
axes[1].set_ylabel('Price ($)')

plt.tight_layout()
plt.show()

## 4. Data Preparation

In [ ]:
# Step 1: Remove extreme price outliers — keep $500 to $150,000
df_model = df[(df['price'] >= 500) & (df['price'] <= 150000)].copy()
print(f'After price filtering: {df_model.shape}')

# Step 2: Filter year to realistic range
df_model = df_model[(df_model['year'] >= 1980) & (df_model['year'] <= 2023)]
print(f'After year filtering: {df_model.shape}')

# Step 3: Drop columns with too many missing values or low utility
# size: 72% missing; VIN, id: non-predictive; region/model: too high cardinality
drop_cols = ['id', 'VIN', 'size', 'region', 'model']
df_model.drop(columns=drop_cols, inplace=True)
print(f'After dropping columns: {df_model.shape}')

# Step 4: Drop rows missing critical features
df_model.dropna(subset=['year', 'odometer', 'manufacturer', 'fuel', 'transmission'], inplace=True)
print(f'After dropping key nulls: {df_model.shape}')

In [ ]:
# Define features and target
target = 'price'
numeric_features = ['year', 'odometer']
categorical_features = ['manufacturer', 'fuel', 'transmission', 'condition',
                         'title_status', 'drive', 'type', 'paint_color', 'state']

X = df_model[numeric_features + categorical_features]
y = df_model[target]

print(f'Feature matrix shape: {X.shape}')
print(f'Target shape: {y.shape}')

In [ ]:
# Train/test split — 80/20
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(f'Train: {X_train.shape} | Test: {X_test.shape}')

In [ ]:
# Build preprocessing pipeline
# Numeric: impute with median -> scale
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# Categorical: impute with most frequent -> one-hot encode
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_transformer, numeric_features),
    ('cat', categorical_transformer, categorical_features)
])

print('Preprocessing pipeline built.')

## 5. Modeling

### 5.1 Linear Regression (Baseline)

In [ ]:
lr_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', LinearRegression())
])

lr_pipeline.fit(X_train, y_train)
y_pred_lr = lr_pipeline.predict(X_test)

rmse_lr = np.sqrt(mean_squared_error(y_test, y_pred_lr))
r2_lr = lr_pipeline.score(X_test, y_test)

print(f'Linear Regression — RMSE: ${rmse_lr:,.0f} | R2: {r2_lr:.4f}')

# 5-fold cross-validation
cv_scores_lr = cross_val_score(lr_pipeline, X_train, y_train,
                                cv=5, scoring='neg_root_mean_squared_error')
print(f'CV RMSE (5-fold): ${-cv_scores_lr.mean():,.0f} +/- ${cv_scores_lr.std():,.0f}')

### 5.2 Ridge Regression with GridSearchCV

In [ ]:
ridge_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', Ridge())
])

ridge_params = {'model__alpha': [0.1, 1.0, 10.0, 100.0, 1000.0]}

ridge_grid = GridSearchCV(
    ridge_pipeline, ridge_params,
    cv=5, scoring='neg_root_mean_squared_error',
    n_jobs=-1, verbose=1
)

ridge_grid.fit(X_train, y_train)

print(f'Best Ridge alpha: {ridge_grid.best_params_}')

y_pred_ridge = ridge_grid.predict(X_test)
rmse_ridge = np.sqrt(mean_squared_error(y_test, y_pred_ridge))
r2_ridge = ridge_grid.score(X_test, y_test)

print(f'Ridge Regression — RMSE: ${rmse_ridge:,.0f} | R2: {r2_ridge:.4f}')

### 5.3 Lasso Regression with GridSearchCV

In [ ]:
lasso_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', Lasso(max_iter=5000))
])

lasso_params = {'model__alpha': [0.1, 1.0, 10.0, 100.0, 1000.0]}

lasso_grid = GridSearchCV(
    lasso_pipeline, lasso_params,
    cv=5, scoring='neg_root_mean_squared_error',
    n_jobs=-1, verbose=1
)

lasso_grid.fit(X_train, y_train)

print(f'Best Lasso alpha: {lasso_grid.best_params_}')

y_pred_lasso = lasso_grid.predict(X_test)
rmse_lasso = np.sqrt(mean_squared_error(y_test, y_pred_lasso))
r2_lasso = lasso_grid.score(X_test, y_test)

print(f'Lasso Regression — RMSE: ${rmse_lasso:,.0f} | R2: {r2_lasso:.4f}')

### 5.4 Model Comparison

In [ ]:
results = pd.DataFrame({
    'Model': ['Linear Regression', 'Ridge Regression', 'Lasso Regression'],
    'RMSE': [rmse_lr, rmse_ridge, rmse_lasso],
    'R2': [r2_lr, r2_ridge, r2_lasso]
}).sort_values('RMSE')

print(results.to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.barplot(data=results, x='Model', y='RMSE', palette='Blues_d', ax=axes[0])
axes[0].set_title('Model Comparison — RMSE (lower is better)', fontweight='bold')
axes[0].set_ylabel('RMSE ($)')
axes[0].tick_params(axis='x', rotation=15)

sns.barplot(data=results, x='Model', y='R2', palette='Greens_d', ax=axes[1])
axes[1].set_title('Model Comparison — R2 (higher is better)', fontweight='bold')
axes[1].set_ylabel('R2 Score')
axes[1].tick_params(axis='x', rotation=15)

plt.tight_layout()
plt.show()

### 5.5 Evaluation Metric Rationale

We use **RMSE (Root Mean Squared Error)** as the primary evaluation metric because:
- It is expressed in the same units as the target variable (dollars), making it directly interpretable
- It penalizes large prediction errors more heavily than MAE — important since large pricing errors cost the dealership real money
- We also report **R²** to show the proportion of price variance explained by the model

### 5.6 Feature Importance — Coefficient Analysis

In [ ]:
# Extract coefficients from the best Ridge model
best_ridge = ridge_grid.best_estimator_

ohe_features = (best_ridge.named_steps['preprocessor']
                .named_transformers_['cat']
                .named_steps['onehot']
                .get_feature_names_out(categorical_features))

all_feature_names = numeric_features + list(ohe_features)
coefficients = best_ridge.named_steps['model'].coef_

coef_df = pd.DataFrame({
    'Feature': all_feature_names,
    'Coefficient': coefficients
})
coef_df['Abs_Coefficient'] = coef_df['Coefficient'].abs()
coef_df = coef_df.sort_values('Abs_Coefficient', ascending=False)

# Plot top 20 features
top20 = coef_df.head(20)
colors = ['tomato' if c < 0 else 'steelblue' for c in top20['Coefficient']]

fig, ax = plt.subplots(figsize=(12, 8))
ax.barh(top20['Feature'], top20['Coefficient'], color=colors)
ax.set_title('Top 20 Features by Coefficient Magnitude (Ridge Regression)',
             fontweight='bold', fontsize=13)
ax.set_xlabel('Coefficient Value (blue = increases price, red = decreases price)')
ax.axvline(0, color='black', linewidth=0.8, linestyle='--')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

print('Top 10 Price-Increasing Features:')
print(coef_df[coef_df['Coefficient'] > 0].head(10)[['Feature', 'Coefficient']].to_string(index=False))
print('\nTop 10 Price-Decreasing Features:')
print(coef_df[coef_df['Coefficient'] < 0].head(10)[['Feature', 'Coefficient']].to_string(index=False))

In [ ]:
# Actual vs Predicted
fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(y_test, y_pred_ridge, alpha=0.2, s=5, color='steelblue')
ax.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()],
        'r--', linewidth=1.5, label='Perfect Prediction')
ax.set_xlabel('Actual Price ($)')
ax.set_ylabel('Predicted Price ($)')
ax.set_title('Actual vs. Predicted Price — Ridge Regression', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

## 6. Findings & Recommendations

---

### Summary for Used Car Dealership

Based on our analysis of over 400,000 used car listings, we identified the key factors that drive used car prices. Below are our findings and actionable recommendations written for a non-technical audience.

---

### Key Findings

**1. Vehicle Age (Year) is the Single Strongest Price Driver**  
Newer vehicles command significantly higher prices. Every additional model year adds meaningfully to resale value. Prioritize recent model years (2015 and newer) in your inventory.

**2. Mileage Strongly Depresses Price**  
Higher odometer readings are strongly associated with lower prices. Low-mileage vehicles (under 50,000 miles) are consistently priced higher. Avoid high-mileage inventory where possible.

**3. Condition Commands a Large Premium**  
Vehicles listed as "new" or "like new" fetch significantly higher prices than "good" or "fair" condition vehicles. Consider reconditioning vehicles before listing to improve their condition grade.

**4. Diesel and Electric Vehicles Command a Price Premium**  
Diesel-powered and electric vehicles tend to sell for more than equivalent gas vehicles, particularly for trucks and SUVs.

**5. 4WD / AWD Vehicles Are Valued Higher**  
Four-wheel drive vehicles command a consistent premium over FWD vehicles — especially in states with harsh winters.

**6. Pickup Trucks and SUVs Lead on Price**  
Trucks and SUVs consistently price higher than sedans and hatchbacks and should be prioritized in inventory.

**7. Clean Title is Non-Negotiable**  
Vehicles with salvage or rebuilt titles are significantly discounted. Stick to clean-title inventory to maximize margins.

---

### Actionable Recommendations

| Priority | Recommendation |
|----------|----------------|
| High | Stock **2015+ model year** vehicles — year is the top price driver |
| High | Prioritize **low-mileage (<50K miles)** inventory |
| High | Focus on **trucks and SUVs** — highest median prices |
| Medium | Invest in **reconditioning** — moving from 'good' to 'like new' adds significant value |
| Medium | Source more **diesel and 4WD vehicles** for premium pricing |
| Lower | Avoid **salvage/rebuilt title** vehicles unless deeply discounted |

---

### Next Steps

1. **Explore non-linear models** (Random Forest, Gradient Boosting) which may capture complex feature interactions and significantly reduce RMSE
2. **Incorporate geographic pricing** — state-level variation was meaningful; a dealership can use this to price competitively in their local market
3. **Build a pricing tool** — deploy this model as a simple internal app where staff can input vehicle details and receive an estimated fair market price
4. **Collect richer data** — number of previous owners, service history, and accident records (Carfax) would likely improve model accuracy substantially